In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

# 1. Establish data tracking paths
data_dir = Path("../data").resolve()
csv_path = data_dir / "SAML-D.csv"

if not csv_path.exists():
    raise FileNotFoundError(f"Missing base transaction data ledger at: {csv_path}")

print(f"Loading contiguous training sequence from {csv_path.name}...")
cols_to_use = ['Time', 'Date', 'Sender_account', 'Receiver_account', 'Amount', 'Is_laundering']
dtypes = {'Sender_account': 'str', 'Receiver_account': 'str', 'Amount': 'float32', 'Is_laundering': 'int8'}

df = pd.read_csv(csv_path, usecols=cols_to_use, dtype=dtypes, nrows=500000)

# 2. Re-establish chronological timeline integrity
df['Timestamp'] = pd.to_datetime(df['Date'] + ' ' + df['Time'])
df.drop(columns=['Date', 'Time'], inplace=True)
df = df.sort_values(by='Timestamp').reset_index(drop=True)

print("Calculating behavioral velocities (using duplicate-safe routing)...")
# Calculate rolling account counts and financial flow values
df['tx_count_1h'] = df.groupby('Sender_account').rolling('1h', on='Timestamp')['Amount'].count().reset_index(level=0, drop=True).sort_index().values
df['tx_amount_sum_1h'] = df.groupby('Sender_account').rolling('1h', on='Timestamp')['Amount'].sum().reset_index(level=0, drop=True).sort_index().values
df['tx_count_24h'] = df.groupby('Sender_account').rolling('24h', on='Timestamp')['Amount'].count().reset_index(level=0, drop=True).sort_index().values

# Safely eliminate rolling lookback boundary NaNs
metrics = ['tx_count_1h', 'tx_amount_sum_1h', 'tx_count_24h']
df[metrics] = df[metrics].fillna(0)

print(f"✅ Feature matrix assembly complete. Records: {df.shape[0]}")
df[['Sender_account', 'Amount'] + metrics + ['Is_laundering']].head()

Loading contiguous training sequence from SAML-D.csv...
Calculating behavioral velocities (using duplicate-safe routing)...
✅ Feature matrix assembly complete. Records: 500000


,Sender_account,Amount,tx_count_1h,tx_amount_sum_1h,tx_count_24h,Is_laundering
0,8724731955,1459.150024,1.0,1459.150024,1.0,0
1,1491989064,6019.640137,1.0,14328.440430,1.0,0
2,287305149,14328.440430,1.0,6019.640137,1.0,0
3,5376652437,11895.000000,1.0,5130.990234,1.0,0
4,9614186178,115.250000,1.0,11895.000000,1.0,0


In [2]:
from sklearn.ensemble import IsolationForest

# 1. Isolate target numerical vectors
feature_cols = ['Amount', 'tx_count_1h', 'tx_amount_sum_1h', 'tx_count_24h']
X = df[feature_cols]
y_true = df['Is_laundering'] # Kept strictly for out-of-sample audit checking

print(f"Training Unsupervised Isolation Forest over features: {feature_cols}")

# 2. Configure the anomaly engine
# contamination: sets the geometric cut-off threshold for the outlier boundary
iso_forest = IsolationForest(
    n_estimators=100,
    contamination=0.005,  # Flagging the top 0.5% strangest records
    random_state=42,
    n_jobs=-1
)

# 3. Fit the engine purely on spatial distribution (No target labels allowed!)
iso_forest.fit(X)

# 4. Extract anomaly predictions and raw anomaly scores
# decision_function returns lower scores for structural anomalies
df['anomaly_score'] = iso_forest.decision_function(X)
raw_predictions = iso_forest.predict(X)

# Map output metrics: Sklearn uses -1 for anomalies and 1 for normal data.
# We conform this to 1 for Anomaly/Laundering and 0 for Normal.
df['y_pred'] = np.where(raw_predictions == -1, 1, 0)

print("✅ Model inference complete! Extracting outlier mappings...")

Training Unsupervised Isolation Forest over features: ['Amount', 'tx_count_1h', 'tx_amount_sum_1h', 'tx_count_24h']
✅ Model inference complete! Extracting outlier mappings...


In [3]:
from sklearn.metrics import classification_report, confusion_matrix

print("=== UNSUPERVISED AML OBSERVABILITY ENGINE AUDIT ===")
cm = confusion_matrix(y_true, df['y_pred'])
print("Confusion Matrix:")
print(f"   Predicted Normal | Predicted Anomaly")
print(f"Actual Normal:   {cm[0][0]} | {cm[0][1]}")
print(f"Actual Laundering: {cm[1][0]} | {cm[1][1]}")

print("\nDetailed Performance Diagnostics:")
print(classification_report(y_true, df['y_pred'], target_names=['Normal', 'Laundering']))

# 5. Inspect a few highly ranked true positive anomalies
print("\n=== SAMPLE TOP-FLAGGED TRUE POSITIVE LAUNDERING ENCOUNTERS ===")
tp_samples = df[(df['Is_laundering'] == 1) & (df['y_pred'] == 1)].head(3)
if not tp_samples.empty:
    print(tp_samples[['Sender_account', 'Amount'] + metrics + ['anomaly_score']])
else:
    print("No true positives captured inside this sample validation boundary.")

=== UNSUPERVISED AML OBSERVABILITY ENGINE AUDIT ===
Confusion Matrix:
   Predicted Normal | Predicted Anomaly
Actual Normal:   496956 | 2488
Actual Laundering: 545 | 11

Detailed Performance Diagnostics:
              precision    recall  f1-score   support

      Normal       1.00      1.00      1.00    499444
  Laundering       0.00      0.02      0.01       556

    accuracy                           0.99    500000
   macro avg       0.50      0.51      0.50    500000
weighted avg       1.00      0.99      1.00    500000


=== SAMPLE TOP-FLAGGED TRUE POSITIVE LAUNDERING ENCOUNTERS ===
       Sender_account        Amount  tx_count_1h  tx_amount_sum_1h  \
88017       207936746  1.610218e+05          1.0      1.610218e+05   
100291     4468415510  1.955586e+05          1.0      1.955586e+05   
151412     5057689301  6.213932e+06          1.0      6.213932e+06   

        tx_count_24h  anomaly_score  
88017            2.0      -0.002372  
100291           1.0      -0.021647  
151412    